# Kopi IoT Web3 - Training Forecasting (best-per-variable)

Versi ini:
1. **Pembersihan outlier** (penyebab utama error besar)
2. **3 model** dibandingkan: Prophet (stabil), Naive, Regresi Linear
3. **Metrik lengkap**: MAE, RMSE, Akurasi (100 - MAPE)
4. **Grafik**: Aktual vs Prediksi, kurva Loss (RMSE) & Akurasi vs jumlah data, perbandingan model
5. **Pilih model TERBAIK otomatis per variabel** lalu **hitung prediksi 14 hari** -> simpan ke model_config.json

Catatan: prediksi 14 hari dihitung di sini (Colab) supaya HF Space ringan & stabil (tidak perlu Prophet/Stan di server).
"Loss" = RMSE, "Akurasi" = 100 - MAPE (persen). Tetap laporkan MAE/RMSE.

Jalankan: Runtime -> Run all.

In [ ]:
!pip -q install prophet

## 1. Ambil data dari ThingSpeak

In [ ]:
import requests, pandas as pd, numpy as np

CHANNEL_ID = 1848413
RESULTS = 8000
TZ = 'Asia/Jakarta'

url = f'https://api.thingspeak.com/channels/{CHANNEL_ID}/feeds.json'
r = requests.get(url, params={'results': RESULTS, 'timezone': TZ}, timeout=30)
data = r.json()
feeds = data['feeds']
print('Channel :', data['channel']['name'])
print('Jumlah feed mentah :', len(feeds))

df = pd.DataFrame(feeds)
df['created_at'] = pd.to_datetime(df['created_at'])
if df['created_at'].dt.tz is not None:
    df['created_at'] = df['created_at'].dt.tz_localize(None)
for col, name in [('field1','suhu'), ('field2','udara'), ('field3','tanah')]:
    df[name] = pd.to_numeric(df[col], errors='coerce')
df = df[['created_at','suhu','udara','tanah']].set_index('created_at')
df.head()

## 2. Bersihkan outlier + resample harian

In [ ]:
RANGES = {'suhu': (10, 45), 'udara': (0, 100), 'tanah': (0, 100)}

for v, (lo, hi) in RANGES.items():
    df[v] = df[v].where((df[v] >= lo) & (df[v] <= hi))

for v in RANGES:
    s = df[v]
    med = s.median()
    mad = (s - med).abs().median()
    if mad and mad > 0:
        batas = 3.5 * 1.4826 * mad
        df[v] = s.where((s - med).abs() <= batas)

harian = df.resample('D').mean()
harian = harian.interpolate(method='time', limit=3, limit_area='inside')
harian = harian.dropna(how='all')

print('Jumlah hari setelah dibersihkan :', len(harian))
harian.describe().round(2)

## 3. Fungsi metrik & util

In [ ]:
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

def mae(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return float(np.mean(np.abs(a - p)))

def rmse(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return float(np.sqrt(np.mean((a - p) ** 2)))

def mape(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    m = np.abs(a) > 1e-6
    if m.sum() == 0:
        return float('nan')
    return float(np.mean(np.abs((a[m] - p[m]) / a[m])) * 100)

def akurasi(a, p):
    return float(max(0.0, 100 - mape(a, p)))

def buat_df(var):
    d = harian[[var]].dropna().reset_index()
    d.columns = ['ds', 'y']
    if d['ds'].dt.tz is not None:
        d['ds'] = d['ds'].dt.tz_localize(None)
    return d

## 4. Latih Prophet (stabil) + baseline, evaluasi di data uji (holdout)

In [ ]:
from prophet import Prophet

VARS = ['suhu', 'udara', 'tanah']
TEST_HARI = 10

hasil, pred_simpan = [], {}

for var in VARS:
    d = buat_df(var)
    if len(d) < 15:
        print('Data', var, 'terlalu sedikit, dilewati'); continue
    k = min(TEST_HARI, max(3, int(len(d) * 0.2)))
    train, test = d.iloc[:-k], d.iloc[-k:]
    ya = test['y'].values

    m = Prophet(weekly_seasonality=False, daily_seasonality=False,
                yearly_seasonality=False, changepoint_prior_scale=0.01)
    m.fit(train)
    yp_prophet = m.predict(test[['ds']])['yhat'].values

    yp_naive = np.repeat(train['y'].values[-1], len(test))

    x = np.arange(len(train))
    coef = np.polyfit(x, train['y'].values, 1)
    xt = np.arange(len(train), len(train) + len(test))
    yp_lin = np.polyval(coef, xt)

    for nama, yp in [('Prophet', yp_prophet), ('Naive', yp_naive), ('Linear', yp_lin)]:
        hasil.append({'variabel': var, 'model': nama,
                      'MAE': round(mae(ya, yp), 3),
                      'RMSE': round(rmse(ya, yp), 3),
                      'Akurasi(%)': round(akurasi(ya, yp), 1)})
    pred_simpan[var] = {'train': train, 'test': test, 'prophet': yp_prophet}

tabel = pd.DataFrame(hasil)
print(tabel.to_string(index=False))
tabel

## 5. Grafik: Aktual vs Prediksi (data uji)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(pred_simpan), 1, figsize=(10, 3.2 * max(1, len(pred_simpan))))
if len(pred_simpan) == 1:
    axes = [axes]
for ax, (var, P) in zip(axes, pred_simpan.items()):
    tr, te = P['train'], P['test']
    ax.plot(tr['ds'], tr['y'], color='#9ca3af', label='Train (aktual)')
    ax.plot(te['ds'], te['y'], 'o-', color='#16a34a', label='Test (aktual)')
    ax.plot(te['ds'], P['prophet'], 's--', color='#ef4444', label='Test (prediksi Prophet)')
    ax.set_title('Aktual vs Prediksi - ' + var)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Kurva Loss (RMSE) & Akurasi vs jumlah data

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
warna = {'suhu': '#ef4444', 'udara': '#3b82f6', 'tanah': '#16a34a'}

for var in VARS:
    d = buat_df(var)
    if len(d) < 20:
        continue
    k = min(TEST_HARI, max(3, int(len(d) * 0.2)))
    train_full, test = d.iloc[:-k], d.iloc[-k:]
    ya = test['y'].values
    sizes = np.unique(np.linspace(max(8, int(len(train_full) * 0.4)),
                                  len(train_full), 6).astype(int))
    xs, losses, accs = [], [], []
    for s in sizes:
        tr = train_full.tail(int(s))
        m = Prophet(weekly_seasonality=False, daily_seasonality=False,
                    yearly_seasonality=False, changepoint_prior_scale=0.01)
        m.fit(tr)
        yp = m.predict(test[['ds']])['yhat'].values
        xs.append(int(s)); losses.append(rmse(ya, yp)); accs.append(akurasi(ya, yp))
    ax1.plot(xs, losses, 'o-', color=warna.get(var), label=var)
    ax2.plot(xs, accs, 'o-', color=warna.get(var), label=var)

ax1.set_title('Loss (RMSE) vs jumlah data latih'); ax1.set_xlabel('jumlah hari'); ax1.set_ylabel('RMSE'); ax1.grid(alpha=0.3); ax1.legend()
ax2.set_title('Akurasi vs jumlah data latih'); ax2.set_xlabel('jumlah hari'); ax2.set_ylabel('Akurasi (100 - MAPE)'); ax2.set_ylim(0, 100); ax2.grid(alpha=0.3); ax2.legend()
plt.tight_layout(); plt.show()

## 7. Perbandingan akurasi antar model

In [ ]:
pivot = tabel.pivot(index='variabel', columns='model', values='Akurasi(%)')
pivot.plot(kind='bar', figsize=(8, 4),
           color={'Prophet': '#16a34a', 'Naive': '#9ca3af', 'Linear': '#3b82f6'})
plt.title('Akurasi (%) per model'); plt.ylabel('Akurasi (100 - MAPE)')
plt.ylim(0, 100); plt.grid(alpha=0.3, axis='y'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
pivot.round(1)

## 8. Pilih model terbaik & HITUNG prediksi 14 hari -> model_config.json

Model dipilih otomatis (MAE terkecil). Untuk tiap variabel, prediksi 14 hari ke depan
dihitung di sini lalu disimpan sebagai angka. HF Space cukup membaca angka ini (tanpa Prophet).

In [ ]:
import json

MAX_H = 14
CLAMP = {'suhu': (0, 60), 'udara': (0, 100), 'tanah': (0, 100)}

terbaik = (tabel.sort_values('MAE')
                .groupby('variabel', as_index=False).first()
                .set_index('variabel'))
print('Model terbaik per variabel (MAE terkecil):')
print(terbaik[['model', 'MAE', 'RMSE', 'Akurasi(%)']].to_string())
print()

def clamp(var, arr):
    lo, hi = CLAMP[var]
    return [round(float(min(hi, max(lo, x))), 2) for x in arr]

config = {
    'info': {
        'dibuat': pd.Timestamp.now().isoformat(timespec='seconds'),
        'last_date': harian.index.max().strftime('%Y-%m-%d'),
        'max_horizon': MAX_H,
        'satuan': {'suhu': 'C', 'udara': '%', 'tanah': '%'},
    },
    'variabel': {},
}

for var in VARS:
    d = buat_df(var)
    metode = terbaik.loc[var, 'model']
    if metode == 'Prophet':
        m = Prophet(weekly_seasonality=False, daily_seasonality=False,
                    yearly_seasonality=False, changepoint_prior_scale=0.01)
        m.fit(d)
        fut = m.make_future_dataframe(periods=MAX_H, freq='D')
        pred = m.predict(fut).tail(MAX_H)['yhat'].tolist()
    elif metode == 'Linear':
        n = len(d)
        coef = np.polyfit(np.arange(n), d['y'].values, 1)
        pred = [float(np.polyval(coef, n - 1 + h)) for h in range(1, MAX_H + 1)]
    else:
        last = float(d['y'].values[-1])
        pred = [last] * MAX_H
    config['variabel'][var] = {'metode': metode,
                               'akurasi': float(terbaik.loc[var, 'Akurasi(%)']),
                               'mae': float(terbaik.loc[var, 'MAE']),
                               'prediksi14': clamp(var, pred)}
    print(var, '->', metode, '(14 hari prediksi siap)')

with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Tersimpan: model_config.json')

tabel.to_json('metrik_backtest.json', orient='records', indent=2)
print('Tersimpan: metrik_backtest.json')

try:
    from google.colab import files
    files.download('model_config.json')
    files.download('metrik_backtest.json')
except Exception as e:
    print('Lewati auto-download:', e)

## 9. Kesimpulan

- Model dipilih otomatis per variabel (best-per-variable), prediksi 14 hari disimpan di **model_config.json**.
- Upload ke HF Space hanya: `app.py`, `requirements.txt`, `README.md`, dan `model_config.json` (TANPA file *_model.json).
- Untuk menyegarkan prediksi (data baru), cukup jalankan ulang notebook ini lalu ganti model_config.json di Space.
- Laporan: sajikan tabel perbandingan + MAE/RMSE/Akurasi; akurasi membaik bila data ditambah (kurva sel 6).